# Домашнее задание № 10. Генерация текста

### Задание 1 (8 баллов).


Возьмите Russian split из вот этого датасета - https://huggingface.co/datasets/CohereLabs/aya_collection_language_split
Он все равно очень большой, поэтому отфильтруйте его до какого-то небольшого рабочего подмножества (например, до 2000 примеров длиной меньше 300 токенов).
Найдите какую-нибудь языковую модель на huggingface, которая не была дообучена на инструкциях (base). Дообучите ее на получившемся датасете.
Возьмите любую задачу из Russian Superglue (например, вот эту - https://russiansuperglue.com/tasks/task_info/MuSeRC) и создайте из нее небольшой оценочный датасет. Формат должен подходить под получившуюся модель, поэтому если в тексте есть отдельно text и question, то вам понадобится их соединить в один промпт. Сделайте предсказания изначальной моделью и дообученой. Посчитайте какую-нибудь метрику качества или даже несколько (например, точное совпадение с правильными ответом + bleu score). Справляется ли дообученная модель лучше? Проанализируйте несколько предсказаний отдельно.
Вы можете использовать модель любого размера и любую технику дообучения (полное дообучение, LoRA или QLoRA)


(*Это задание сложнее предыдущих, поэтому не стесняйтесь задавать вопросы в чате или лично)


### Задание 2 (2 балла)
Два дополнительных балла можно получить если размер модели больше 3B.

In [1]:
!pip3 install -q transformers "datasets<4.0.0" peft trl accelerate scikit-learn bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.0 MB/s eta 0:00:00


In [2]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### 1. Загрузка и исследование обучающих данных

In [3]:
from datasets import load_dataset
import pandas as pd

aya = load_dataset(
    "CohereLabs/aya_collection_language_split",
    "russian",
    split="train")
print(aya)

README.md: 0.00B [00:00, ?B/s]

russian/train-00000-of-00002.parquet:   0%|          | 0.00/704M [00:00<?, ?B/s]

russian/train-00001-of-00002.parquet:   0%|          | 0.00/739M [00:00<?, ?B/s]

russian/validation-00000-of-00001.parque(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

russian/test-00000-of-00001.parquet:   0%|          | 0.00/145M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4005166 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/322325 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/338994 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'inputs', 'targets', 'dataset_name', 'sub_dataset_name', 'task_type', 'template_id', 'language', 'script', 'split'],
    num_rows: 4005166
})


In [4]:
print(aya.features)
print()
print(aya[0])

{'id': Value(dtype='int64', id=None), 'inputs': Value(dtype='string', id=None), 'targets': Value(dtype='string', id=None), 'dataset_name': Value(dtype='string', id=None), 'sub_dataset_name': Value(dtype='string', id=None), 'task_type': Value(dtype='string', id=None), 'template_id': Value(dtype='int64', id=None), 'language': Value(dtype='string', id=None), 'script': Value(dtype='string', id=None), 'split': Value(dtype='string', id=None)}

{'id': 1, 'inputs': 'Турецкий народ (), или турки (), также известные как анатолийские турки (), являются тюркской этнической группой и нацией, живущей в основном в Турции и говорящей на турецком языке, наиболее распространенном тюркском языке. Они являются крупнейшей этнической группой в Турции, а также, безусловно, крупнейшей этнической группой среди носителей тюркских языков. Этнические турецкие меньшинства существуют на бывших землях Османской империи. Кроме того, с современной миграцией была создана турецкая диаспора, особенно в Западной Европе.  

In [5]:
# Получим представление, о какой длине текстов идёт речь
lengths = [len(ex["inputs"]) + len(ex["targets"])
           for ex in aya.select(range(10000))]
pd.Series(lengths).describe()

,0
count,10000.000000
mean,1942.135800
std,925.576467
min,295.000000
25%,976.000000
50%,2135.500000
75%,2664.000000
max,8521.000000


In [6]:
# берём случайный пул для последующей фильтрации по токенам
aya_subset = aya.shuffle(seed=42).select(range(50000))

In [7]:
# Типы задач
pd.Series(aya_subset["task_type"]).value_counts()

,count
dialogue,14864
text-simplification,12468
event-linking,12022
question-answering,7157
summarization,1577
generation,1351
paraphrase-identification,552
-,9


### 2. Загрузка и исследование оценочного датасета

In [8]:
muserc = load_dataset(
    "RussianNLP/russian_super_glue",
    "muserc",
    trust_remote_code=True)
print(muserc)

README.md: 0.00B [00:00, ?B/s]

russian_super_glue.py: 0.00B [00:00, ?B/s]

data/MuSeRC.zip:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11950 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2235 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7614 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['paragraph', 'question', 'answer', 'idx', 'label'],
        num_rows: 11950
    })
    validation: Dataset({
        features: ['paragraph', 'question', 'answer', 'idx', 'label'],
        num_rows: 2235
    })
    test: Dataset({
        features: ['paragraph', 'question', 'answer', 'idx', 'label'],
        num_rows: 7614
    })
})


In [9]:
val_data = muserc["validation"]
print(val_data.features)
print()
print(val_data[0])

{'paragraph': Value(dtype='string', id=None), 'question': Value(dtype='string', id=None), 'answer': Value(dtype='string', id=None), 'idx': {'paragraph': Value(dtype='int32', id=None), 'question': Value(dtype='int32', id=None), 'answer': Value(dtype='int32', id=None)}, 'label': ClassLabel(names=['False', 'True'], id=None)}

{'paragraph': '(1) Самый первый «остров» Архипелага возник в 1923 году на месте Соловецкого монастыря. (2) Затем появились ТОНы — тюрьмы особого назначения и этапы. (3) Люди попадали на Архипелаг разными способами: в вагон-заках, на баржах, пароходах и пешими этапами. (4) В тюрьмы арестованных доставляли в «воронках» — фургончиках чёрного цвета. (5) Роль портов Архипелага играли пересылки, временные лагеря, состоящие из палаток, землянок, бараков или участков земли под открытым небом. (6) На всех пересылках держать «политических» в узде помогали специально отобранные урки, или «социально близкие». (7) Солженицын побывал на пересылке Красная Пресня в 1945 году. (8) Эм

In [10]:
idx = val_data["idx"]
para_ids = [x["paragraph"] for x in idx]
question_ids = [x["question"] for x in idx]

print(f"Всего примеров:          {len(val_data)}")
print(f"Уникальных параграфов:   {len(set(para_ids))}")
print(f"Уникальных вопросов:     {len(set(zip(para_ids, question_ids)))}")
print()
print("Распределение меток (0=нет, 1=да):")
pd.Series(val_data["label"]).value_counts().rename({0: "0 (нет)", 1: "1 (да)"})

Всего примеров:          2235
Уникальных параграфов:   100
Уникальных вопросов:     529

Распределение меток (0=нет, 1=да):


,count
0 (нет),1242
1 (да),993


> Если честно, я не до конца понял прикол этого датасета, типа — в карточке написано, что это бинарная классификация, но это не данетка, то есть тут ответы на вопросы, правильные и неправильные. Поэтому я буду спрашивать, является ли ответ на вопрос правильным согласно прочитанному тексту — кажется, в этом и состоит идея.

In [11]:
# для оценки берём сбалансированный сплит из validation
val_subset = val_data.shuffle(seed=42).select(range(300))
print(f"Оценочная выборка: {len(val_subset)}")
pd.Series(val_subset["label"]).value_counts().rename(
    {0: "0 (нет)", 1: "1 (да)"})

Оценочная выборка: 300


,count
0 (нет),166
1 (да),134


### 3. Форматирование данных

Обучающий датасет просто приведём к виду сообщений чата с ролями user и assistant, а к оценочному датасету добавим промпт-инструкцию, он будет без ответов. В сообщение его оформим уже на этапе оценки.

In [12]:
def format_aya(example):
    return {"messages": [
        {"role": "user", "content": example["inputs"]},
        {"role": "assistant", "content": example["targets"]}
    ]}


aya_formatted = aya_subset.map(format_aya)
print(aya_formatted[0]["messages"])

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

[{'content': 'Правда ли следующее утверждение? "Внешние Гебриды и Сьерра-Леоне имеют общую береговую линию". Цепочка мысли и решения для этого вопроса:', 'role': 'user'}, {'content': 'Эти два места находятся в совершенно разных частях мира. Ответ - нет.', 'role': 'assistant'}]


In [13]:
def format_muserc_prompt(example):
    return (
        'Прочитай текст и определи: верен ли данный ответ на вопрос? '
        'Ответь только "да" или "нет".\n\n'
        f'Текст: {example["paragraph"]}\n'
        f'Вопрос: {example["question"]}\n'
        f'Ответ: {example["answer"]}'
    )


print(format_muserc_prompt(val_data[0]))

Прочитай текст и определи: верен ли данный ответ на вопрос? Ответь только "да" или "нет".

Текст: (1) Самый первый «остров» Архипелага возник в 1923 году на месте Соловецкого монастыря. (2) Затем появились ТОНы — тюрьмы особого назначения и этапы. (3) Люди попадали на Архипелаг разными способами: в вагон-заках, на баржах, пароходах и пешими этапами. (4) В тюрьмы арестованных доставляли в «воронках» — фургончиках чёрного цвета. (5) Роль портов Архипелага играли пересылки, временные лагеря, состоящие из палаток, землянок, бараков или участков земли под открытым небом. (6) На всех пересылках держать «политических» в узде помогали специально отобранные урки, или «социально близкие». (7) Солженицын побывал на пересылке Красная Пресня в 1945 году. (8) Эмигранты, крестьяне и «малые народы» перевозили красными эшелонами. (9) Чаще всего такие эшелоны останав­ливались на пустом месте, посреди степи или тайги, и осуждённые сами строили лагерь. (10) Особо важные заключённые, в основном учёные, пер

### 4. Загрузка базовой модели

Будем использовать Qwen3-4B-base. Как показала практика, модель спокойно влезает без квантизации для инференса, но для обучения, увы, только QLoRa. НО: никто не мешает нам обучить адаптор в QLoRa, а потом для инференса приклеить его к полной, неквантизованной модели (привет, Unsloth!) — поэтому оценивать мы будем модель без квантизации, потом перегрузим с квантизацией для файнтьюнинга, а потом снова загрузим полный чекпойнт и прилепим к нему наш адаптор.

In [14]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-4B-Base"

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print(f"dtype: {next(model.parameters()).dtype}")
mem_gb = sum(p.numel() * p.element_size()
             for p in model.parameters()) / 1024**3
print(f"memory footprint: ~{mem_gb:.2f} GB")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

dtype: torch.bfloat16
memory footprint: ~7.49 GB


### 5. Фильтрация по длине промпта в оценочном датасете

Чтобы не вышло так, что модель будет видеть только короткие сэмплы, а на этапе инференса будут длинные, отфильтруем наш обучающий датасет так, чтобы максимальное количество токенов совпадало с таковым в датасете, на котором потом будем оцениваться.

In [17]:
# длины промптов в оценочном датасете (формат инференса)
muserc_lengths = []
for ex in val_subset:
    messages = [{"role": "user", "content": format_muserc_prompt(ex)}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    muserc_lengths.append(len(tokenizer(text)["input_ids"]))

pd.Series(muserc_lengths).describe()

,0
count,300.000000
mean,621.346667
std,142.899969
min,432.000000
25%,542.750000
50%,599.000000
75%,651.250000
max,1444.000000


In [18]:
MAX_LEN = max(muserc_lengths)
print(f"Максимальная длина в MuSeRC: {MAX_LEN} токенов")

Максимальная длина в MuSeRC: 1444 токенов


In [19]:
# Фильтруем обучающий датасет по реальной длине токенов (полный chat
# format с ответом)
def add_token_length(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"token_length": len(tokenizer(text)["input_ids"])}


aya_with_lengths = aya_formatted.map(add_token_length)
aya_refiltered = aya_with_lengths.filter(
    lambda x: x["token_length"] <= MAX_LEN)
aya_refiltered = aya_refiltered.shuffle(seed=42).select(
    range(min(2000, len(aya_refiltered))))
print(f"После фильтрации: {len(aya_refiltered)} примеров")
pd.Series(aya_refiltered["token_length"]).describe()

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

После фильтрации: 2000 примеров


,0
count,2000.000000
mean,169.623000
std,158.340556
min,33.000000
25%,102.000000
50%,127.000000
75%,164.000000
max,1177.000000


Забавно: длинных инструкций в этом датасете нет в принципе, в оценочном и то есть длиннее. Можно было и не фильтровать, а взять случайные 2000.

### 6. Оцениваем базовую модель

In [20]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm.notebook import tqdm


def evaluate_muserc(model, tokenizer, dataset):
    model.eval()
    predictions, labels = [], []

    for example in tqdm(dataset):
        messages = [{"role": "user", "content": format_muserc_prompt(example)}]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(
            new_tokens, skip_special_tokens=True).strip().lower()

        if len(predictions) < 10:
            print(
                f"  [{len(predictions)+1}] метка={example['label']}  ответ={response!r}")

        if "да" in response[:20]:
            pred = 1
        elif "нет" in response[:20]:
            pred = 0
        else:
            pred = None  # модель не следует инструкции

        predictions.append(pred)
        labels.append(example["label"])

    return predictions, labels

In [21]:
preds_base, labels_eval = evaluate_muserc(model, tokenizer, val_subset)

preds_base_flat = [p if p is not None else -1 for p in preds_base]

print("\nБазовая модель:")
print(
    f"  не ответила да/нет: {sum(p is None for p in preds_base)}/{len(preds_base)}")
print(f"  accuracy: {accuracy_score(labels_eval, preds_base_flat):.3f}")
print(
    f"  F1:       {f1_score(labels_eval, preds_base_flat, average='macro', labels=[0, 1], zero_division=0):.3f}")

  0%|          | 0/300 [00:00<?, ?it/s]

  [1] метка=0  ответ='assistant\nassistant\nassistant\nassistant\nassistant'
  [2] метка=0  ответ='прочитай текст и определи:'
  [3] метка=1  ответ='прочитай текст и определи:'
  [4] метка=1  ответ='assistant\nassistant\nassistant\nassistant\nassistant'
  [5] метка=0  ответ='прочитай текст и определи:'
  [6] метка=0  ответ='assistant\nassistant\nassistant\nassistant\nassistant'
  [7] метка=1  ответ='assistant\nпрочитай текст и опред'
  [8] метка=1  ответ='assistant\nassistant\nassistant\nassistant\nassistant'
  [9] метка=0  ответ='прочитай текст и определи:'
  [10] метка=0  ответ='прочитай текст и определи:'

Базовая модель:
  не ответила да/нет: 276/300
  accuracy: 0.063
  F1:       0.119


Модель просто не следует инструкциям, а продолжает текст, даже не пытаясь в подавляющем большинстве случаев ответить "да" или "нет". Скорее всего, результат мог бы быть лучше, если бы в базовой модели мы не использовали chat_template, а просто бы подали сырой текст, закончив промпт чем-то вроде "ответ: " — тогда бы, наверное, она бы смогла продолжить токеном "да" или "нет" — следовать шаблону чата в претрейне она просто не может в виду отсутствия соответствующих обучающих примеров в её данных.

In [24]:
import gc

del model

gc.collect()
torch.cuda.empty_cache()

print(
    f"Занято видеопамяти после очистки: {torch.cuda.memory_allocated() / 1024**3:.2f} ГБ")

Занято видеопамяти после очистки: 0.01 ГБ


### 7. Настройка LoRA

Теперь уже загрузим квантизованную модель и настроим LoRa

In [25]:
# Теперь загружаем модель с квантованием
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"dtype: {next(model.parameters()).dtype}")
mem_gb = sum(p.numel() * p.element_size()
             for p in model.parameters()) / 1024**3
print(f"memory footprint: ~{mem_gb:.2f} GB")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

dtype: torch.bfloat16
memory footprint: ~4.11 GB


In [26]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157


### 8. Обучаем

Две эпохи Колабу оказалось вполне по силам.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=aya_refiltered,
    args=SFTConfig(
        output_dir="qwen3_lora",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=2e-4,
        bf16=True,
        warmup_ratio=0.05,
        weight_decay=0.01,
        logging_steps=25,
        save_strategy="epoch",
        max_length=MAX_LEN,
        neftune_noise_alpha=5,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
import gc

model.config.use_cache = False
gc.collect()
torch.cuda.empty_cache()

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
25,2.192607
50,1.644048
75,1.601419
100,1.593918
125,1.567697
150,1.592121
175,1.557677
200,1.540853
225,1.578521
250,1.599582


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
25,2.192607
50,1.644048
75,1.601419
100,1.593918
125,1.567697
150,1.592121
175,1.557677
200,1.540853
225,1.578521
250,1.599582


TrainOutput(global_step=500, training_loss=1.4996163024902345, metrics={'train_runtime': 8459.8152, 'train_samples_per_second': 0.473, 'train_steps_per_second': 0.059, 'total_flos': 1.5060780944437248e+16, 'train_loss': 1.4996163024902345})

Лосс на трейне не очень стабильно идёт вниз и колеблется, видимо, из-за малого количества обучающих данных — но в итоге всё же снижается.

### 9. Вливаем адаптор в полную модель

Сохраняем адаптор, удаляем квантизованную модель, перезагружаем полную и присоединяем к ней наш адаптор.

In [ ]:
# Сохраняем финальный адаптор
adapter_path = "./temp_final_adapter"
trainer.model.save_pretrained(adapter_path)

# Удаляем квантованную модель и тренер
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

print(
    f"Занято видеопамяти после очистки: {torch.cuda.memory_allocated() / 1024**3:.2f} ГБ")

Занято видеопамяти после очистки: 1.47 ГБ


In [ ]:
from peft import PeftModel

# Перезагружаем базовую модель
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Прикручиваем к ней наш сохраненный адаптер
peft_model = PeftModel.from_pretrained(base_model, adapter_path)

# Сливаем веса адаптера с базовой моделью
merged_model = peft_model.merge_and_unload()
merged_model.config.use_cache = True

print(
    f"Занято видеопамяти после загрузки и слияния: {torch.cuda.memory_allocated() / 1024**3:.2f} ГБ")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Занято видеопамяти после загрузки и слияния: 8.96 ГБ


### 10. Оценка итоговой модели

In [ ]:
preds_final, _ = evaluate_muserc(merged_model, tokenizer, val_subset)

  0%|          | 0/300 [00:00<?, ?it/s]

  [1] метка=0  ответ='לחלוט\n\n нет, нет. это не имеет с'
  [2] метка=0  ответ='לחלוט\n\n нет, нет. это не то,'
  [3] метка=1  ответ='לחלוט\n\naincontri\n\nда, конечно. это'
  [4] метка=1  ответ='לחלוט\n\n\tndrfc\n\nда, конечно. вот'
  [5] метка=0  ответ='לחלוט\n\n нет, нет. это не то,'
  [6] метка=0  ответ='לחלוט\n\n нет, нет. это не то,'
  [7] метка=1  ответ='לחלוט\n\n\tndrfc\n\nда, конечно. это'
  [8] метка=1  ответ='לחלוט\n\naincontri\n\nнет, нет. это'
  [9] метка=0  ответ='לחלוט\n\n нет, нет. вопрос:'
  [10] метка=0  ответ='לחלוט\n\n нет, нет. это не то,'


In [ ]:
preds_final_flat = [p if p is not None else -1 for p in preds_final]

print("Сравнение:")
print(
    f"  не ответила да/нет: {sum(p is None for p in preds_base)}/{len(preds_base)}"
    f" -> {sum(p is None for p in preds_final)}/{len(preds_final)}")
print(f"  accuracy:  {accuracy_score(labels_eval, preds_base_flat):.3f} -> {accuracy_score(labels_eval, preds_final_flat):.3f}")
print(
    f"  F1:        {f1_score(labels_eval, preds_base_flat, average='macro', labels=[0, 1], zero_division=0):.3f}"
    f" -> {f1_score(labels_eval, preds_final_flat, average='macro', labels=[0, 1], zero_division=0):.3f}")

Сравнение:
  не ответила да/нет: 282/300 -> 14/300
  accuracy:  0.043 -> 0.847
  F1:        0.084 -> 0.867


Прежде всего, модель стала очень хорошо следовать инструкциям, и теперь по крайней мере продолжает текст так, как нужно — то есть примеров хватило, чтобы модель поймала правильный формат. Метрики тоже очень хорошие, то есть понимание текста хорошее, и верные от неверных вопросов отличаются достаточно хорошо. По примерам ответов видно, что модель не может самостоятельно остановиться, пытается сгенерировать какие-то объяснения и продолжить рассуждать, но в целом правильно отвечает на поставленный вопрос, и мы можем извлечь "да" или "нет".